In [67]:
# Use helpers
import sys
import os

target_dir = os.path.abspath('..') 

if target_dir not in sys.path:
    sys.path.append(target_dir)

In [68]:
# Use helpers
from preprocess import * 
from understatapi import UnderstatClient
import torch as nn

understat = UnderstatClient()

In [69]:
# Fit model to Erling Haaland for POC

stats = ["goals", "xG", "assists", "xA", "key_passes", "xGChain", "xGBuildup"]
window_size = 10

f_stats = get_player_stats_df(understat, '8260', stats, window_size)

# set date to index
f_stats.set_index('date', inplace=True)
f_stats.head()

,rolling_goals_per_90,rolling_xG_per_90,rolling_assists_per_90,rolling_xA_per_90,rolling_key_passes_per_90,rolling_xGChain_per_90,rolling_xGBuildup_per_90
date,,,,,,,
2020-05-23,1.304348,0.786954,0.260870,0.134672,1.043478,1.040308,0.384068
2020-05-26,0.864198,0.623131,0.246914,0.127467,0.987654,0.860065,0.360656
2020-06-13,0.734694,0.479098,0.244898,0.126426,0.979592,0.714099,0.357711
2020-06-17,0.481283,0.376565,0.240642,0.116004,0.842246,0.607090,0.283868
2020-06-20,0.721925,0.642953,0.240642,0.175442,0.962567,0.898064,0.243283


In [70]:
# train/test split
train_len = int(len(f_stats) * 0.8)
train_df = f_stats.iloc[:train_len]
test_df = f_stats.iloc[train_len:]

In [71]:
from torch.utils.data import Dataset
import torch

class CustomFootballDataset(Dataset):
    def __init__(self, stats_df, window_size):
        self.stats_df = stats_df
        self.window_size = window_size

    def __len__(self):
        return len(self.stats_df) - self.window_size

    def __getitem__(self, idx):
        row = self.stats_df.iloc[idx:idx + self.window_size]
        label = self.stats_df.iloc[idx + self.window_size]
        row_tensor = torch.tensor(row.values, dtype=torch.float32)
        label_tensor = torch.tensor(label.values, dtype=torch.float32)
        return row_tensor, label_tensor

In [72]:
# create datasets
train_dataset = CustomFootballDataset(train_df, window_size)
test_dataset = CustomFootballDataset(test_df, window_size)

In [73]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=32,
    shuffle=True
)

test_dataloader = DataLoader(
    dataset=test_dataset,
    batch_size=32,
    shuffle=False
)

In [74]:
for features, labels in train_dataloader:
    print(features.shape)
    print(labels.shape)
    break

torch.Size([32, 10, 7])
torch.Size([32, 7])


In [75]:
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_hidden_state = lstm_out[:, -1, :]
        output = self.fc(last_hidden_state)
        return output

In [76]:
import torch.optim as optim

model = LSTMModel(input_size=len(f_stats.columns), hidden_size=64, output_size=len(f_stats.columns))
optimizer = optim.Adam(model.parameters(), lr=0.0001)
loss_fn = nn.MSELoss()

In [79]:
for input, target in train_dataloader:
    optimizer.zero_grad()
    output = model(input)
    loss = loss_fn(output, target)
    loss.backward()
    optimizer.step()

In [80]:
model.compile()

In [81]:
model.eval()
with torch.no_grad():
    for input, target in test_dataloader:
        output = model(input)
        loss = loss_fn(output, target)
        print(f'Test Loss: {loss.item()}')

Test Loss: 0.3907979428768158
